# Sales Forecasting with PyCaret
## Dr José M Albornoz
### February 2025

This notebook presents an example of how Pycaret can be used for time series forecasting. The dataset used in this example describe retail sales at a department store; no exogenous variables are considered.

# 0.- Imports

In [44]:
from pycaret.time_series import *
import pandas as pd
import numpy as np
import plotly.express as px
import datetime

# maximum number of rdataframe ows and columns displayed
pd.set_option('display.max_rows', 50000)
pd.set_option('display.max_columns', 500)

pd.options.mode.chained_assignment = None

RANDOM_SEED = 801

# 1.- Load data

In [2]:
data = pd.read_csv('../data/sales_without_exogenous.csv')
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)
target = "Sales"
data.head()

,Sales
Date,
2012-07-01,638368
2012-07-02,667057
2012-07-03,569246
2012-07-04,599165
2012-07-05,626678


In [3]:
data.shape

(714, 1)

In [4]:
data.dtypes

Sales    int64
dtype: object

In [5]:
type(data.index)

pandas.core.indexes.datetimes.DatetimeIndex

## 1.1.- Plot series 

In [1]:
# plot using PyCaret's in-built plotting capability
#data.plot()

In [2]:
#fig = px.line(data, x=data.index, y="Sales", template = 'plotly_dark')
#fig.show()

# 2.- PyCaret Setup

The `setup` function initializes the training environment and creates the transformation pipeline. Setup function must be called before executing any other function in PyCaret. Setup has only one required parameter i.e. `data`. All the other parameters are optional.

In [8]:
# We want to forecast the next 12 months of data and we will use 3 fold cross-validation to test the models.
fh = 14 # or alternately fh = np.arange(1,13)
fold = 3 # (default)

In [9]:
s = setup(data, fh = 14, session_id = RANDOM_SEED)

,Description,Value
0,session_id,801
1,Target,Sales
2,Approach,Univariate
3,Exogenous Variables,Not Present
4,Original data shape,"(714, 1)"
5,Transformed data shape,"(714, 1)"
6,Transformed train set shape,"(700, 1)"
7,Transformed test set shape,"(14, 1)"
8,Rows with missing values,0.0%
9,Fold Generator,ExpandingWindowSplitter


# 3.- Exploratory data analysis

## 3.1.- Series statistics

The `check_stats` function is used to get summary statistics and run statistical tests on the original data, transformed data, or model residuals.

In [10]:
# check statistical tests on original data
check_stats()

,Test,Test Name,Data,Property,Setting,Value
0,Summary,Statistics,Transformed,Length,,714.0
1,Summary,Statistics,Transformed,# Missing Values,,0.0
2,Summary,Statistics,Transformed,Mean,,714285.62465
3,Summary,Statistics,Transformed,Median,,641247.0
4,Summary,Statistics,Transformed,Standard Deviation,,240068.101949
5,Summary,Statistics,Transformed,Variance,,57632693573.286682
6,Summary,Statistics,Transformed,Kurtosis,,7.7919
7,Summary,Statistics,Transformed,Skewness,,2.115388
8,Summary,Statistics,Transformed,# Distinct Values,,712.0
9,White Noise,Ljung-Box,Transformed,Test Statictic,"{'alpha': 0.05, 'K': 24}",2730.801087


## 3.2.- Exploratory plots

In [3]:
#s.plot_model(data_kwargs={"plot_data_type": ["original", "transformed"]}, fig_kwargs={'height': 600, "width": 1200})

The above plot shows that no transformation has been performed on the series. Let us now examine the autocorrelation and partial autocorrelation plots.

In [4]:
# ACF plot
#s.plot_model(plot="acf", fig_kwargs={'height': 600, "width": 1200})

In [5]:
#s.plot_model(plot="pacf", data_kwargs={'nlags':36}, fig_kwargs={'height': 600, "width": 1200})

Both the ACF and PACF plots show that we are dealing with a non-random series. There is evidence of non-stationarity in these plots as both ACF and PACF decay slowly as lags increase; this non-stationarity is also evident in the series plot and in the statistics for the series. Long-time dependences can be seen: 7-days seasonality as well as a 35-days are observed both in the ACF and PACF plots. 

Next, we will examine the periodogram and the FFT plots.

In [6]:
#s.plot_model(plot="periodogram", fig_kwargs={'height': 600, "width": 1200})

In [7]:
#s.plot_model(plot="fft", fig_kwargs={'height': 600, "width": 1200})

Yearly and weekly seasonality can be seen in peaks at f = 0.0028 (1/0.0028 ~ 357) and f = 0.1429 (1/0/1429 ~ 7).

Alternatively, the diagnostics plot includes all the plots we have examined so far plus histogram and Q-Q plot.

In [8]:
#s.plot_model(plot="diagnostics", fig_kwargs={"height": 600, "width": 1200})

As previously noted, the series requires differencing for stationarity. We'll now examine periodograms and FFT plots after applying 1-day and 7-day differencing.

In [9]:
# s.plot_model(
#     plot="diff",
#     data_kwargs={"lags_list": [[1], [7]], "acf": True, "pacf": True, "periodogram": True},
#     fig_kwargs={"height": 800, "width": 1500}
# )

We observe that a 1-day differencing is enough to make the series stationary, as evidenced by the quick decay seen in the ACF and PACF plots.

Let's generate another classical diagnostic plot: time series decomposition:

In [10]:
# By default the seasonal period is the one detected during setup 
#s.plot_model(plot="decomp", data_kwargs={'seasonal_period': 35}, fig_kwargs={"height": 800, "width": 1200})

An important consideration in time series modelling is the temporal sequence of the data points in the series: the series cannot be split randomly as this means that we could be introducing target leakage by using future data to predict past data. For this reason, it is very important to maintain temporal sequence when splitting the data. This is achieved using backtesting. The following figures show the train-test split in the dataset and the backtesting splits within the dataset.

In [11]:
# Show the train-test splits on the dataset
#s.plot_model(plot="train_test_split", fig_kwargs={"height": 800, "width": 1100})

In [12]:
# Show the backtesting splits inside the train set
# The blue dots represent the training data for each fold.
# The orange dots represent the validation data for each fold
#s.plot_model(plot="cv", fig_kwargs={"height": 400, "width": 1000})

# 4.- Available models

The following function lists all the time series models available for training.

In [21]:
# check available models
models()

,Name,Reference,Turbo
ID,,,
naive,Naive Forecaster,sktime.forecasting.naive.NaiveForecaster,True
grand_means,Grand Means Forecaster,sktime.forecasting.naive.NaiveForecaster,True
snaive,Seasonal Naive Forecaster,sktime.forecasting.naive.NaiveForecaster,True
polytrend,Polynomial Trend Forecaster,sktime.forecasting.trend._polynomial_trend_for...,True
arima,ARIMA,sktime.forecasting.arima.ARIMA,True
auto_arima,Auto ARIMA,sktime.forecasting.arima.AutoARIMA,True
exp_smooth,Exponential Smoothing,sktime.forecasting.exp_smoothing.ExponentialSm...,True
ets,ETS,sktime.forecasting.ets.AutoETS,True
theta,Theta Forecaster,sktime.forecasting.theta.ThetaForecaster,True


The `compare_models` function trains and evaluates the performance of all the estimators available in the model library using cross-validation. The output of this function is a scoring grid with average cross-validated scores. Metrics evaluated during CV can be accessed using the get_metrics function. Custom metrics can be added or removed using add_metric and remove_metric function.

You can use the `include` and `exclude` parameter in the `compare_models` to train only selected models or exclude specific models from training by passing the model id's in `exclude` parameter.

In [22]:
#best = compare_models(sort='MASE', include = ['naive', 'snaive', 'arima', 'exp_smooth', 'xgboost_cds_dt', 'dt_cds_dt'])

In [23]:
best = compare_models(sort='MASE', include = ['naive', 'snaive', 'arima', 'exp_smooth', 'xgboost_cds_dt', 'dt_cds_dt', 'bats', 'tbats'])

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
tbats,TBATS,0.4293,0.4492,81822.0540,130748.8009,0.1088,0.1156,-0.0167,1926.2567
xgboost_cds_dt,Extreme Gradient Boosting w/ Cond. Deseasonalize & Detrending,0.4853,0.5189,92639.0076,151159.7160,0.1255,0.1308,-0.5938,36.4300
arima,ARIMA,0.5292,0.5000,100972.6338,145536.5017,0.1363,0.1460,-0.2464,87.4267
snaive,Seasonal Naive Forecaster,0.5296,0.4959,101043.9286,144307.5079,0.1363,0.1459,-0.2004,9.3200
exp_smooth,Exponential Smoothing,0.5379,0.5170,102756.5996,150641.5859,0.1424,0.1490,-0.6354,124.7100
bats,BATS,0.5823,0.5389,111379.1940,157137.6243,0.1601,0.1583,-1.1173,598.6667
dt_cds_dt,Decision Tree w/ Cond. Deseasonalize & Detrending,0.6550,0.7007,125692.6968,205120.7285,0.1971,0.1711,-6.5442,24.3200
naive,Naive Forecaster,0.7727,0.6467,148099.1190,188741.7380,0.2308,0.2169,-2.8610,30.3967


In [25]:
#best = compare_models(sort='MASE', include = ['naive', 'snaive', 'arima', 'exp_smooth', 'xgboost_cds_dt', 'dt_cds_dt', 'grand_means',
#                                              'polytrend', 'ets', 'theta', 'stlf', 'croston', 'bats', 'tbats'])

In [26]:
best

TBATS(show_warnings=False, sp=[35], use_box_cox=True)

The function above has return the trained model object as an output. If you need access to the scoring grid you can use `pull` function to access the dataframe.

The score on the test set can be accessed using predict_model on the best performing model:

In [27]:
prediction_holdout = predict_model(best)

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,TBATS,0.4594,0.3533,86734.8662,102337.4083,0.1593,0.1444,-3.4496


# 5.- Analyse model

A first step in analysing the performance of the BATS model is to examine the residuals. For a well-fitting time series model the residuals should resemble white noise, meaning that the model has captured all the systematic information in the data, and only random fluctuations remain. In the ideal case, the residuals should be normally distributed, with a mean close to zero and without any significant autocorrelation.

In [28]:
# check_stats on residuals of best model
check_stats(estimator = best)

,Test,Test Name,Data,Property,Setting,Value
0,Summary,Statistics,Residual,Length,,700.0
1,Summary,Statistics,Residual,# Missing Values,,0.0
2,Summary,Statistics,Residual,Mean,,194.291499
3,Summary,Statistics,Residual,Median,,-9770.4389
4,Summary,Statistics,Residual,Standard Deviation,,163997.328242
5,Summary,Statistics,Residual,Variance,,26895123670.437389
6,Summary,Statistics,Residual,Kurtosis,,24.201187
7,Summary,Statistics,Residual,Skewness,,2.191345
8,Summary,Statistics,Residual,# Distinct Values,,700.0
9,White Noise,Ljung-Box,Residual,Test Statictic,"{'alpha': 0.05, 'K': 24}",24.965758


You can use the plot_model function to analyzes the performance of a trained model on the test set. It may require re-training the model in certain cases.

In [13]:
# residuals plot
#plot_model(best, plot = 'residuals', fig_kwargs={"height": 400, "width": 1100})

To further assess the fit provided by the model, we will plot the forecast for the specified 14 days horizon against the actual data: 

In [14]:
# plot forecast
#plot_model(best, plot = 'forecast', fig_kwargs={"height": 400, "width": 1100})

In [15]:
# plot forecast for the next 35 days
#plot_model(best, plot = 'forecast', data_kwargs = {'fh' : 35})

# 6.- Forecasting

As seen in the previous section, 

In [47]:
# predict on test set
holdout_pred = predict_model(best)

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,TBATS,0.4594,0.3533,86734.8662,102337.4083,0.1593,0.1444,-3.4496


In [48]:
# show predictions df
holdout_pred.head()

,y_pred
2014-06-01,658444.9578
2014-06-02,602605.1216
2014-06-03,548974.7180
2014-06-04,543429.7602
2014-06-05,611864.2533


In [ ]:
# generate forecast for 36 period in future
predict_model(best, fh = 36)

# 6.- Save model

In [ ]:
# save pipeline
save_model(best, 'time_series_v0')

In [ ]:
# load pipeline
loaded_best_pipeline = load_model('time_series_v0')
loaded_best_pipeline

# 7.- Experiment Logging

PyCaret integrates with many different type of experiment loggers (default = 'mlflow'). To turn on experiment tracking in PyCaret you can set `log_experiment` and `experiment_name` parameter. It will automatically track all the metrics, hyperparameters, and artifacts based on the defined logger.

In [ ]:
# s = setup(data, fh = 3, session_id = 123, log_experiment='mlflow', experiment_name='airline_experiment')

In [ ]:
# compare models
# best = compare_models()

In [ ]:
# start mlflow server on localhost:5000
# !mlflow ui

By default PyCaret uses MLFlow logger; that can be changed using `log_experiment` parameter. Following loggers are available:

- mlflow
- wandb
- comet_ml
- dagshub
  
For more information check out the docstring of the setup function.

In [ ]:
help(setup)

# 8.- Create model

This function trains and evaluates the performance of a given estimator using cross-validation. The output of this function is a scoring grid with CV scores by fold. Metrics evaluated during CV can be accessed using the `get_metrics` function. Custom metrics can be added or removed using `add_metric` and `remove_metric` function. All the available models can be accessed using the `models` function.

In [ ]:
# train ets with default fold=3
ets = create_model('ets')

The function above has return trained model object as an output. The scoring grid is only displayed and not returned. If you need access to the scoring grid you can use `pull` function to access the dataframe.

In [ ]:
ets_results = pull()
print(type(ets_results))
ets_results

In [ ]:
# train theta model with fold=5
theta = create_model('theta', fold=5)

In [ ]:
# train theta with specific model parameters
create_model('theta', deseasonalize = False, fold=5)

Some other parameters that you might find very useful in create_model are:

- cross_validation
- engine
- fit_kwargs
  
You can check the docstring of the function for more info.

# 9.- Tune model

The `tune_model` function tunes the hyperparameters of the model. The output of this function is a scoring grid with cross-validated scores by fold. The best model is selected based on the metric defined in `optimize` parameter. Metrics evaluated during cross-validation can be accessed using the `get_metrics` function. Custom metrics can be added or removed using `add_metric` and `remove_metric` function.

Metric to optimize can be defined in `optimize` parameter (default = 'MASE'). Also, a custom tuned grid can be passed with `custom_grid` parameter.

In [ ]:
# train a dt model with default params
dt = create_model('dt_cds_dt')

In [ ]:
# tune hyperparameters of best model 
tuned_dt = tune_model(dt)

In [ ]:
# define tuning grid
dt_grid = {'regressor__max_depth' : [None, 2, 4, 6, 8, 10, 12]}

# tune model with custom grid and metric = MAE
tuned_dt = tune_model(dt, custom_grid = dt_grid, optimize = 'MAE')

In [ ]:
# see tuned_best params
tuned_dt

In [ ]:
# to access the tuner object you can set return_tuner = True
tuned_best, tuner = tune_model(dt, return_tuner=True)

In [ ]:
# model object
tuned_best

In [ ]:
# tuner object
tuner

For more details on all available search_library and search_algorithm please check the docstring. Some other parameters that you might find very useful in tune_model are:

- choose_better
- custom_scorer
- n_iter
- search_algorithm
- optimize
- 
You can check the docstring of the function for more info.

In [ ]:
help(tune_model)

# 10.- Blend models

This function trains a `EnsembleForecaster` for select models passed in the `estimator_list` parameter. The output of this function is a scoring grid with CV scores by fold. Metrics evaluated during CV can be accessed using the `get_metrics` function. Custom metrics can be added or removed using `add_metric` and `remove_metric` function.

In [ ]:
# top 3 models based on mae
best_mae_models_top3

In [ ]:
# blend top 3 models
blend_models(best_mae_models_top3)

Some other parameters that you might find very useful in blend_models are:

- choose_better
- method
- weights
- fit_kwargs
- optimize
- 
You can check the docstring of the function for more info.

In [ ]:
help(blend_models)

# 11.- Plot model

In [ ]:
# plot acf
# for certain plots you don't need a trained model
plot_model(plot = 'acf')

In [ ]:
# plot diagnostics
# for certain plots you don't need a trained model
plot_model(plot = 'diagnostics')

Some other parameters that you might find very useful in plot_model are:

- fig_kwargs
- data_kwargs
- display_format
- return_fig
- return_data
- save

You can check the docstring of the function for more info.

In [ ]:
help(plot_model)

# 12.- Finalise model

This function trains a given model on the entire dataset including the hold-out set.

In [ ]:
final_best = finalize_model(best)

In [ ]:
final_best

# 13.- Deploy model

This function deploys the entire ML pipeline on the cloud.

AWS: When deploying model on AWS S3, environment variables must be configured using the command-line interface. To configure AWS environment variables, type aws configure in terminal. The following information is required which can be generated using the Identity and Access Management (IAM) portal of your amazon console account:

- AWS Access Key ID
- AWS Secret Key Access
- Default Region Name (can be seen under Global settings on your AWS console)
- Default output format (must be left blank)

GCP: To deploy a model on Google Cloud Platform ('gcp'), the project must be created using the command-line or GCP console. Once the project is created, you must create a service account and download the service account key as a JSON file to set environment variables in your local environment. Learn more about it: https://cloud.google.com/docs/authentication/production

Azure: To deploy a model on Microsoft Azure ('azure'), environment variables for the connection string must be set in your local environment. Go to settings of storage account on Azure portal to access the connection string required. AZURE_STORAGE_CONNECTION_STRING (required as environment variable) Learn more about it: https://docs.microsoft.com/en-us/azure/storage/blobs/storage-quickstart-blobs-python?toc=%2Fpython%2Fazure%2FTOC.json

In [ ]:
# deploy model on aws s3
# deploy_model(best, model_name = 'my_first_platform_on_aws',
#             platform = 'aws', authentication = {'bucket' : 'pycaret-test'})

In [ ]:
# load model from aws s3
# loaded_from_aws = load_model(model_name = 'my_first_platform_on_aws', platform = 'aws',
#                              authentication = {'bucket' : 'pycaret-test'})

# loaded_from_aws

# 14.- Save/load experiment

This function saves all the experiment variables on disk, allowing to later resume without rerunning the setup function.

In [ ]:
# save experiment
save_experiment('my_experiment')

In [ ]:
# load experiment from disk
exp_from_disk = load_experiment('my_experiment', data=data)